# OpenSSL Cryptography Lab on Google Colab

This notebook demonstrates:
1. AES-256 encryption and decryption
2. RSA-2048 public-key encryption and decryption
3. ECDSA P-256 digital signing and signature verification

> Note: ECDSA is a digital-signature algorithm, not an encryption algorithm. RSA is used for public-key encryption here.

## 0. Check OpenSSL version

In [1]:
!openssl version

OpenSSL 3.0.13 30 Jan 2024 (Library: OpenSSL 3.0.13 30 Jan 2024)


# 1. AES-256 Encryption and Decryption

In [2]:
plaintext = "Hello, this is a secret message using AES-256."

with open("plaintext.txt", "w") as f:
    f.write(plaintext)

print("Original plaintext:")
print(open("plaintext.txt").read())

Original plaintext:
Hello, this is a secret message using AES-256.


### Encrypt with AES-256-CBC

In [3]:
!openssl enc -aes-256-cbc \
    -salt \
    -pbkdf2 \
    -iter 100000 \
    -in plaintext.txt \
    -out aes_encrypted.bin \
    -pass pass:123456

### Display AES ciphertext as Base64

In [4]:
!openssl base64 -in aes_encrypted.bin

U2FsdGVkX1/Fnb4HkOm7RPi1a5tgyA+w3J2amU2S5TM8n6hTcnHx6LfPHESgRb0b
ovmKewmXfLNUGfIxSCXj1Q==


### Decrypt AES ciphertext

In [5]:
!openssl enc -d -aes-256-cbc \
    -pbkdf2 \
    -iter 100000 \
    -in aes_encrypted.bin \
    -out aes_decrypted.txt \
    -pass pass:123456

!echo "Decrypted plaintext:"
!cat aes_decrypted.txt

Decrypted plaintext:
Hello, this is a secret message using AES-256.

# 2. RSA-2048 Public-Key Encryption and Decryption

### Generate RSA-2048 private and public keys

In [6]:
!openssl genpkey \
    -algorithm RSA \
    -pkeyopt rsa_keygen_bits:2048 \
    -out rsa_private.pem

!openssl pkey \
    -in rsa_private.pem \
    -pubout \
    -out rsa_public.pem

!ls -lh rsa_private.pem rsa_public.pem

...+.....+....+.....+.+...+..+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*....+....+...........+.+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*..+.......+..+.+.........+..+.....................+..........+...+......+...+.....+.+..+...............+...+....+..+.+........................+...+..+.+.....+....+......+.....+............+...............+.+.....+............+...+......+.+...+......+.....+.+........+.........................+..+.......+.....+................+.........+.....+.+......+...+...+..............+.+...+..+..................+.+...........+.........+...+...+....+.....+......+...+...................+...+...+.....+.+.....+.+..+...+....+...+...+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
..+.+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*........+......+...........+...+...+.............+..+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*...+...+.+.....+......+.........+....+....

### Create RSA plaintext

In [7]:
with open("rsa_plaintext.txt", "w") as f:
    f.write("Secret message encrypted using RSA-2048.")

print(open("rsa_plaintext.txt").read())

Secret message encrypted using RSA-2048.


### Encrypt using the RSA public key with OAEP + SHA-256

In [8]:
!openssl pkeyutl \
    -encrypt \
    -pubin \
    -inkey rsa_public.pem \
    -in rsa_plaintext.txt \
    -out rsa_encrypted.bin \
    -pkeyopt rsa_padding_mode:oaep \
    -pkeyopt rsa_oaep_md:sha256 \
    -pkeyopt rsa_mgf1_md:sha256

### Display RSA ciphertext as Base64

In [9]:
!openssl base64 -in rsa_encrypted.bin

ETw0UOtiBrhQcE2BySTZdIfXuKO7bbik3QLpILssRsEns1Xw4SI4QDG7zs83a/Kq
ICQlxTR1gZrU4oz9Xe+ccKWIVMuS1mdQqZyCxcwLSfNnI1sxgchfaclt0rhhc86P
X/O5pbizkh1ib5yWhZ7Y77HY7Zbws8hApreEbBwRTEKOVMPh6lI44gwjJMTjFrBp
1Igybc/JFH4eNxWy1/iak8i2bhtwTVGRexBc0bjOzmOFGJrmhRvkFxhh3X3doYd7
Dv88Srxjj3Ipd+1J/S6X631f7JbC/H+fLCjesAONbeZwb23Sj4gHn/5ocvSwQojO
+ZQ0B4G+6Wi2bFNMueQ/nA==


### Decrypt using the RSA private key

In [10]:
!openssl pkeyutl \
    -decrypt \
    -inkey rsa_private.pem \
    -in rsa_encrypted.bin \
    -out rsa_decrypted.txt \
    -pkeyopt rsa_padding_mode:oaep \
    -pkeyopt rsa_oaep_md:sha256 \
    -pkeyopt rsa_mgf1_md:sha256

!echo "Decrypted RSA plaintext:"
!cat rsa_decrypted.txt

Decrypted RSA plaintext:
Secret message encrypted using RSA-2048.

# 3. ECDSA P-256 Digital Signature and Verification

### Generate ECDSA P-256 private and public keys

In [11]:
!openssl genpkey \
    -algorithm EC \
    -pkeyopt ec_paramgen_curve:P-256 \
    -out ecdsa_private.pem

!openssl pkey \
    -in ecdsa_private.pem \
    -pubout \
    -out ecdsa_public.pem

!ls -lh ecdsa_private.pem ecdsa_public.pem

-rw------- 1 root root 241 Sep 15 08:28 ecdsa_private.pem
-rw-r--r-- 1 root root 178 Sep 15 08:28 ecdsa_public.pem


### Create a document to sign

In [12]:
with open("document.txt", "w") as f:
    f.write("This document will be digitally signed.")

print(open("document.txt").read())

This document will be digitally signed.


### Sign the document using ECDSA + SHA-256

In [13]:
!openssl dgst \
    -sha256 \
    -sign ecdsa_private.pem \
    -out signature.bin \
    document.txt

### Display signature as Base64

In [14]:
!openssl base64 -in signature.bin

MEYCIQDv1DETaZohSz/Yhrq4sF7ea+FBBGX45ig/ePSt+Efj9wIhAKAJtb9vOEnq
Pd7t8bNrGhes+b4Esx0fqjoiZva/cVAD


### Verify the signature using the ECDSA public key

In [15]:
!openssl dgst \
    -sha256 \
    -verify ecdsa_public.pem \
    -signature signature.bin \
    document.txt

Verified OK


In [16]:
# ============================================================
# PART II - PYTHON
# 1. AES-256 Decryption
# 2. RSA-2048 Decryption
# 3. ECDSA P-256 Signature Verification
# ============================================================

!pip install -q cryptography

from cryptography.hazmat.primitives import hashes, serialization, padding
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.exceptions import InvalidSignature


# ============================================================
# 1. AES-256-CBC DECRYPTION
# OpenSSL input file: aes_encrypted.bin
# Password used in Part I: 123456
# ============================================================

print("=" * 60)
print("1. AES-256-CBC DECRYPTION")
print("=" * 60)

PASSWORD = b"123456"

with open("aes_encrypted.bin", "rb") as f:
    encrypted_data = f.read()

# OpenSSL salted format:
# Salted__ | 8-byte salt | ciphertext
if encrypted_data[:8] != b"Salted__":
    raise ValueError("Invalid OpenSSL encrypted file format")

salt = encrypted_data[8:16]
ciphertext = encrypted_data[16:]

# OpenSSL PBKDF2:
# SHA-256, 100000 iterations
# AES-256 key = 32 bytes
# CBC IV = 16 bytes
kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=48,
    salt=salt,
    iterations=100000
)

key_iv = kdf.derive(PASSWORD)

aes_key = key_iv[:32]
iv = key_iv[32:48]

cipher = Cipher(
    algorithms.AES(aes_key),
    modes.CBC(iv)
)

decryptor = cipher.decryptor()

padded_plaintext = (
    decryptor.update(ciphertext)
    + decryptor.finalize()
)

# Remove PKCS#7 padding
unpadder = padding.PKCS7(128).unpadder()

aes_plaintext = (
    unpadder.update(padded_plaintext)
    + unpadder.finalize()
)

print("Salt       :", salt.hex())
print("AES Key    :", aes_key.hex())
print("IV         :", iv.hex())
print("Plaintext  :", aes_plaintext.decode())


# ============================================================
# 2. RSA-2048 DECRYPTION
# OpenSSL files:
#   rsa_private.pem
#   rsa_encrypted.bin
#
# Padding used:
#   RSA-OAEP
#   SHA-256
#   MGF1-SHA256
# ============================================================

print("\n" + "=" * 60)
print("2. RSA-2048 DECRYPTION")
print("=" * 60)

with open("rsa_private.pem", "rb") as f:
    rsa_private_key = serialization.load_pem_private_key(
        f.read(),
        password=None
    )

with open("rsa_encrypted.bin", "rb") as f:
    rsa_ciphertext = f.read()

rsa_plaintext = rsa_private_key.decrypt(
    rsa_ciphertext,
    asym_padding.OAEP(
        mgf=asym_padding.MGF1(
            algorithm=hashes.SHA256()
        ),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Ciphertext size:", len(rsa_ciphertext), "bytes")
print("Plaintext      :", rsa_plaintext.decode())


# ============================================================
# 3. ECDSA P-256 SIGNATURE VERIFICATION
# OpenSSL files:
#   ecdsa_public.pem
#   signature.bin
#   document.txt
#
# Algorithm:
#   ECDSA + SHA-256
# ============================================================

print("\n" + "=" * 60)
print("3. ECDSA P-256 SIGNATURE VERIFICATION")
print("=" * 60)

with open("ecdsa_public.pem", "rb") as f:
    ecdsa_public_key = serialization.load_pem_public_key(
        f.read()
    )

with open("signature.bin", "rb") as f:
    signature = f.read()

with open("document.txt", "rb") as f:
    document = f.read()

print("Document:", document.decode())

try:
    ecdsa_public_key.verify(
        signature,
        document,
        ec.ECDSA(hashes.SHA256())
    )

    print("Signature verification: VALID")

except InvalidSignature:
    print("Signature verification: INVALID")


print("\n" + "=" * 60)
print("ALL OPERATIONS COMPLETED")
print("=" * 60)

1. AES-256-CBC DECRYPTION
Salt       : c59dbe0790e9bb44
AES Key    : d5cc395acdd296897afb3b95423fc097a343ebce596297c4ec85dcfb93173acf
IV         : 6cd959c9787026f375674146651ac9e8
Plaintext  : Hello, this is a secret message using AES-256.

2. RSA-2048 DECRYPTION
Ciphertext size: 256 bytes
Plaintext      : Secret message encrypted using RSA-2048.

3. ECDSA P-256 SIGNATURE VERIFICATION
Document: This document will be digitally signed.
Signature verification: VALID

ALL OPERATIONS COMPLETED
